# Teste Colab

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices('GPU'))

# Instalação Bibliotecas

In [ ]:
!pip install -q tensorflow numpy matplotlib pandas scikit-learn pillow opencv-python

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

print("Bibliotecas carregadas com sucesso!")

# Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/TCC_Soja_IA"

DATASET_DIR = os.path.join(BASE_DIR, "dataset")
TESTE_DIR = os.path.join(BASE_DIR, "teste")
RESULTADOS_DIR = os.path.join(BASE_DIR, "resultados")

print(DATASET_DIR)

In [ ]:
import os

classes = ["Saudavel", "Ferrugem", "Mancha_Alvo"]

for classe in classes:
    caminho = os.path.join(DATASET_DIR, classe)

    arquivos = os.listdir(caminho)

    print(f"{classe}: {len(arquivos)} imagens")

In [ ]:
for classe in classes:
    caminho = os.path.join(TESTE_DIR, classe)

    arquivos = os.listdir(caminho)

    print(f"{classe} - teste: {len(arquivos)} imagens")

# Visualização Imagens

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
import random

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for i, classe in enumerate(classes):

    pasta = os.path.join(DATASET_DIR, classe)
    arquivos = os.listdir(pasta)

    arquivo = random.choice(arquivos)

    caminho = os.path.join(pasta, arquivo)

    imagem = Image.open(caminho)

    axes[i].imshow(imagem)
    axes[i].set_title(classe)
    axes[i].axis("off")

plt.tight_layout()
plt.show()

# Conjunto de Treinamento e Validação

*Carregamento das imagens*

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

print("Classes:", train_ds.class_names)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10)
])

*EfficientNetB0 (ImageNet)*

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

*Classificador*

In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    3,
    activation="softmax"
)(x)

model = tf.keras.Model(inputs, outputs)

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Treinamento

*20 épocas*

In [ ]:
EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

*Gráficos do Treinamento*

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history["accuracy"], label="Treinamento")
plt.plot(history.history["val_accuracy"], label="Validação")

plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.title("Acurácia durante o treinamento")
plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history["loss"], label="Treinamento")
plt.plot(history.history["val_loss"], label="Validação")

plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Loss durante o treinamento")
plt.legend()
plt.grid()

plt.show()

*Modelo*

In [ ]:
MODEL_PATH = os.path.join(
    RESULTADOS_DIR,
    "modelo_soja_efficientnetb0.keras"
)

model.save(MODEL_PATH)

print("Modelo salvo em:", MODEL_PATH)

# Teste

In [ ]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    TESTE_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(test_ds.class_names)

# Avaliação do Modelo

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Acurácia no teste:", test_accuracy)

In [ ]:
import numpy as np

y_true = []
y_pred = []

for imagens, etiquetas in test_ds:

    previsoes = model.predict(imagens, verbose=0)

    classes_preditas = np.argmax(previsoes, axis=1)

    y_true.extend(etiquetas.numpy())
    y_pred.extend(classes_preditas)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

*Relatório de Classificação*

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=test_ds.class_names
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7,6))

plt.imshow(cm)

plt.xticks(
    range(len(test_ds.class_names)),
    test_ds.class_names,
    rotation=45
)

plt.yticks(
    range(len(test_ds.class_names)),
    test_ds.class_names
)

plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.title("Matriz de confusão")

plt.colorbar()

plt.show()

# Resultados Modelo

In [ ]:
import os

SEVERIDADE_DIR = os.path.join(
    RESULTADOS_DIR,
    "severidade"
)

OVERLAY_DIR = os.path.join(
    SEVERIDADE_DIR,
    "overlays"
)

os.makedirs(SEVERIDADE_DIR, exist_ok=True)
os.makedirs(OVERLAY_DIR, exist_ok=True)

print("Pasta criada:")
print(SEVERIDADE_DIR)

*Teste de Leitura de Imagem*

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

arquivos_teste = test_ds.file_paths

print("Total de imagens de teste:", len(arquivos_teste))

caminho = arquivos_teste[0]

imagem = Image.open(caminho).convert("RGB")

plt.figure(figsize=(6, 6))
plt.imshow(imagem)
plt.axis("off")
plt.title(os.path.basename(caminho))
plt.show()

# Cálculo de Severidade

*Formatação de Imagens*

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def criar_mascara_folha(img_rgb):

    img = img_rgb.copy()

    # Converter para HSV
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)

    # Saturação
    saturation = hsv[:, :, 1]

    # Valor/brilho
    value = hsv[:, :, 2]

    # Pixel que não seja praticamente branco
    mask = (
        (saturation > 20) |
        (value < 245)
    )

    mask = mask.astype(np.uint8) * 255

    # Remover pequenos ruídos
    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    # Encontrar componentes
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)

    if num_labels > 1:
        # Ignorar o fundo e pegar o maior componente
        maior = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = np.where(
            labels == maior,
            255,
            0
        ).astype(np.uint8)

    return mask

In [ ]:
mask_folha = criar_mascara_folha(
    np.array(imagem)
)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(imagem)
plt.title("Imagem original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask_folha, cmap="gray")
plt.title("Máscara da folha")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def aplicar_mascara_folha(img_rgb, mask):

    resultado = np.zeros_like(img_rgb)

    resultado[mask > 0] = img_rgb[mask > 0]

    return resultado

In [ ]:
folha_recortada = aplicar_mascara_folha(
    np.array(imagem),
    mask_folha
)

plt.figure(figsize=(6, 6))
plt.imshow(folha_recortada)
plt.axis("off")
plt.title("Folha isolada")
plt.show()

*Cálculo da Área Foliar Total*

In [ ]:
area_folha = np.sum(mask_folha > 0)

print("Área da folha em pixels:", area_folha)

*Identificação da Área Foliar Afetada*

In [ ]:
def indice_verde(img_rgb):

    img = img_rgb.astype(np.float32)

    R = img[:, :, 0]
    G = img[:, :, 1]
    B = img[:, :, 2]

    gi = G / (R + G + B + 1e-6)

    return gi

In [ ]:
gi = indice_verde(np.array(imagem))

plt.figure(figsize=(7, 6))
plt.imshow(gi, cmap="gray")
plt.colorbar()
plt.title("Índice de verde")
plt.axis("off")
plt.show()

In [ ]:
LIMIAR_VERDE = 0.38

In [ ]:
mask_sintoma = (
    (gi < LIMIAR_VERDE) &
    (mask_folha > 0)
)

mask_sintoma = mask_sintoma.astype(np.uint8) * 255

In [ ]:
kernel = np.ones((3, 3), np.uint8)

mask_sintoma = cv2.morphologyEx(
    mask_sintoma,
    cv2.MORPH_OPEN,
    kernel
)

mask_sintoma = cv2.morphologyEx(
    mask_sintoma,
    cv2.MORPH_CLOSE,
    kernel
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(imagem)
plt.title("Imagem original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(mask_sintoma, cmap="gray")
plt.title("Região identificada como sintomática")
plt.axis("off")

plt.tight_layout()
plt.show()

*Cálculo da Área Foliar Afetada*

In [ ]:
area_afetada = np.sum(mask_sintoma > 0)

print("Área total da folha:", area_folha)
print("Área afetada:", area_afetada)

*Cálculo de Severidade*

In [ ]:
severidade = (
    area_afetada /
    area_folha
) * 100

print(f"Severidade estimada: {severidade:.2f}%")

In [ ]:
def criar_overlay(img_rgb, mask_sintoma):

    overlay = img_rgb.copy()

    # Máscara booleana
    mask = mask_sintoma > 0

    # Destacar em vermelho
    overlay[mask] = [255, 0, 0]

    # Misturar imagem original e destaque
    resultado = cv2.addWeighted(
        img_rgb,
        0.65,
        overlay,
        0.35,
        0
    )

    return resultado

*Resultado*

In [ ]:
overlay = criar_overlay(
    np.array(imagem),
    mask_sintoma
)

plt.figure(figsize=(7, 7))
plt.imshow(overlay)
plt.title(f"Severidade estimada: {severidade:.2f}%")
plt.axis("off")
plt.show()

*Teste*

In [ ]:
classes_testes = {}

for caminho in test_ds.file_paths:

    classe = os.path.basename(
        os.path.dirname(caminho)
    )

    if classe not in classes_testes:
        classes_testes[classe] = caminho

print(classes_testes)

In [ ]:
for classe, caminho in classes_testes.items():

    img = np.array(
        Image.open(caminho).convert("RGB")
    )

    mask_folha = criar_mascara_folha(img)

    gi = indice_verde(img)

    mask_sintoma = (
        (gi < LIMIAR_VERDE) &
        (mask_folha > 0)
    ).astype(np.uint8) * 255

    area_folha = np.sum(mask_folha > 0)
    area_afetada = np.sum(mask_sintoma > 0)

    severidade = (
        area_afetada / area_folha
    ) * 100

    overlay = criar_overlay(img, mask_sintoma)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title(f"{classe}\nOriginal")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(mask_sintoma, cmap="gray")
    plt.title("Área afetada")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title(f"Severidade: {severidade:.2f}%")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

*Calibração Limiar*

In [ ]:
import os
import random

random.seed(42)

imagens_calibracao = {}

for classe in classes:

    pasta = os.path.join(DATASET_DIR, classe)

    arquivos = [
        f for f in os.listdir(pasta)
        if f.lower().endswith(
            (".jpg", ".jpeg", ".png", ".bmp", ".webp")
        )
    ]

    selecionadas = random.sample(arquivos, 10)

    imagens_calibracao[classe] = [
        os.path.join(pasta, f)
        for f in selecionadas
    ]

for classe in imagens_calibracao:
    print(classe, ":", len(imagens_calibracao[classe]))

In [ ]:
def gerar_mascara_sintoma(img_rgb, mask_folha, limiar):

    gi = indice_verde(img_rgb)

    mask_sintoma = (
        (gi < limiar) &
        (mask_folha > 0)
    ).astype(np.uint8) * 255

    kernel = np.ones((3, 3), np.uint8)

    mask_sintoma = cv2.morphologyEx(
        mask_sintoma,
        cv2.MORPH_OPEN,
        kernel
    )

    mask_sintoma = cv2.morphologyEx(
        mask_sintoma,
        cv2.MORPH_CLOSE,
        kernel
    )

    return mask_sintoma

In [ ]:
imagens_exemplo = {
    classe: imagens_calibracao[classe][0]
    for classe in classes
}

for classe, caminho in imagens_exemplo.items():
    print(classe, "→", caminho)

In [ ]:
limiares = [0.30, 0.34, 0.38, 0.42, 0.46, 0.50]

for classe, caminho in imagens_exemplo.items():

    img = np.array(
        Image.open(caminho).convert("RGB")
    )

    mask_folha = criar_mascara_folha(img)

    fig, axes = plt.subplots(
        2, 3,
        figsize=(12, 8)
    )

    for ax, limiar in zip(axes.ravel(), limiares):

        mask_sintoma = gerar_mascara_sintoma(
            img,
            mask_folha,
            limiar
        )

        ax.imshow(mask_sintoma, cmap="gray")
        ax.set_title(f"Limiar = {limiar}")
        ax.axis("off")

    plt.suptitle(
        f"{classe} — comparação dos limiares",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd

resultados_severidade = []

*Limiar Definido = 0.38*

In [ ]:

# Processamento final

import os
import random
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

# Caminhos

BASE_DIR = Path("/content/drive/MyDrive/TCC_Soja_IA")

DATASET_DIR = BASE_DIR / "dataset"
TESTE_DIR = BASE_DIR / "teste"
RESULTADOS_DIR = BASE_DIR / "resultados"

SEVERIDADE_DIR = RESULTADOS_DIR / "severidade"
CALIBRACAO_DIR = SEVERIDADE_DIR / "calibracao"
OVERLAY_DIR = SEVERIDADE_DIR / "overlays"
MASCARAS_DIR = SEVERIDADE_DIR / "mascaras"
FIGURAS_DIR = SEVERIDADE_DIR / "figuras"

classes = ["Saudavel", "Ferrugem", "Mancha_Alvo"]

EXTENSOES = (
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
)

# Verificação acesso ao Drive

if not BASE_DIR.is_dir():

    raise FileNotFoundError(
        "Pasta TCC_Soja_IA não encontrada. "
        "Verifique se o Google Drive está conectado."
    )

# Criar pastas de resultados
for pasta in [
    SEVERIDADE_DIR,
    CALIBRACAO_DIR,
    OVERLAY_DIR,
    MASCARAS_DIR,
    FIGURAS_DIR
]:

    pasta.mkdir(
        parents=True,
        exist_ok=True
    )

# Listar imagens

def listar_imagens(pasta):

    pasta = Path(pasta)

    return sorted([
        arquivo
        for arquivo in pasta.iterdir()
        if arquivo.is_file()
        and arquivo.suffix.lower() in EXTENSOES
    ])

# Máscara da folha

def criar_mascara_folha(img_rgb):

    hsv = cv2.cvtColor(
        img_rgb,
        cv2.COLOR_RGB2HSV
    )

    saturation = hsv[:, :, 1]
    value = hsv[:, :, 2]

    mask = (
        (saturation > 20) |
        (value < 245)
    ).astype(np.uint8) * 255

    kernel = np.ones(
        (5, 5),
        np.uint8
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    num_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(mask)
    )

    if num_labels > 1:

        maior = 1 + np.argmax(
            stats[1:, cv2.CC_STAT_AREA]
        )

        mask = np.where(
            labels == maior,
            255,
            0
        ).astype(np.uint8)

    return mask

# Índice de verde

def indice_verde(img_rgb):

    img = img_rgb.astype(np.float32)

    R = img[:, :, 0]
    G = img[:, :, 1]
    B = img[:, :, 2]

    gi = G / (R + G + B + 1e-6)

    return gi

# Máscara dos sintomas

def gerar_mascara_sintoma(
    img_rgb,
    mask_folha,
    limiar
):

    gi = indice_verde(img_rgb)

    mask_sintoma = (
        (gi < limiar) &
        (mask_folha > 0)
    ).astype(np.uint8) * 255

    kernel = np.ones(
        (3, 3),
        np.uint8
    )

    mask_sintoma = cv2.morphologyEx(
        mask_sintoma,
        cv2.MORPH_OPEN,
        kernel
    )

    mask_sintoma = cv2.morphologyEx(
        mask_sintoma,
        cv2.MORPH_CLOSE,
        kernel
    )

    # Restringir a área sintomática à folha
    mask_sintoma[mask_folha == 0] = 0

    return mask_sintoma

# Overlay

def criar_overlay(img_rgb, mask_sintoma):

    overlay = img_rgb.copy()

    mask = mask_sintoma > 0

    overlay[mask] = [255, 0, 0]

    resultado = cv2.addWeighted(
        img_rgb,
        0.65,
        overlay,
        0.35,
        0
    )

    return resultado

# Limiar definido

LIMIAR_FINAL = 0.38

# Conferência quantidade de imagens

for classe in classes:

    n_dataset = len(
        listar_imagens(DATASET_DIR / classe)
    )

    n_teste = len(
        listar_imagens(TESTE_DIR / classe)
    )

    print(
        f"{classe}: "
        f"{n_dataset} desenvolvimento | "
        f"{n_teste} teste"
    )

    assert n_dataset == 300
    assert n_teste == 20

print("\nLimiar definido:", LIMIAR_FINAL)

print("\nPreparação concluída com sucesso!")

*Conferência Final Limiar*

In [ ]:
# Verificação das 30 imagens de calibração

random.seed(42)

registros_calibracao = []

for classe in classes:

    arquivos = listar_imagens(
        DATASET_DIR / classe
    )

    selecionadas = random.sample(
        arquivos,
        10
    )

    # Duas figuras com cinco imagens cada
    for bloco in range(2):

        grupo = selecionadas[
            bloco * 5:(bloco + 1) * 5
        ]

        fig, axes = plt.subplots(
            5,
            4,
            figsize=(14, 17)
        )

        for i, caminho in enumerate(grupo):

            img = np.array(
                Image.open(caminho).convert("RGB")
            )

            mask_folha = criar_mascara_folha(img)

            mask_sintoma = gerar_mascara_sintoma(
                img,
                mask_folha,
                LIMIAR_FINAL
            )

            area_folha = np.count_nonzero(
                mask_folha
            )

            area_afetada = np.count_nonzero(
                mask_sintoma
            )

            severidade = (
                area_afetada / area_folha * 100
                if area_folha > 0
                else np.nan
            )

            overlay = criar_overlay(
                img,
                mask_sintoma
            )

            # Imagem original
            axes[i, 0].imshow(img)
            axes[i, 0].set_title("Original")

            # Máscara da folha
            axes[i, 1].imshow(
                mask_folha,
                cmap="gray"
            )
            axes[i, 1].set_title("Máscara da folha")

            # Máscara dos sintomas
            axes[i, 2].imshow(
                mask_sintoma,
                cmap="gray"
            )
            axes[i, 2].set_title("Área sintomática")

            # Overlay
            axes[i, 3].imshow(overlay)
            axes[i, 3].set_title(
                f"Severidade: {severidade:.2f}%"
            )

            for ax in axes[i]:
                ax.axis("off")

            registros_calibracao.append({
                "classe": classe,
                "imagem": caminho.name,
                "caminho": str(caminho),
                "severidade_estimada": severidade
            })

        plt.suptitle(
            f"Calibração - {classe} - Parte {bloco + 1}",
            fontsize=15
        )

        plt.tight_layout()

        plt.savefig(
            CALIBRACAO_DIR /
            f"calibracao_{classe}_{bloco + 1}.png",
            dpi=200,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)

# Salvar quais imagens foram utilizadas
df_calibracao = pd.DataFrame(
    registros_calibracao
)

df_calibracao.to_csv(
    CALIBRACAO_DIR / "imagens_calibracao.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Total de imagens verificadas:", len(df_calibracao))
print("Limiar utilizado:", LIMIAR_FINAL)
print("Imagens de calibração salvas com sucesso!")

*Cálculo Severidade Teste*

In [ ]:
# Cálculo Severidade - 60 imagens


resultados_severidade = []

for classe in classes:

    arquivos = listar_imagens(
        TESTE_DIR / classe
    )

    # Pastas para armazenar resultados
    pasta_overlay = OVERLAY_DIR / classe
    pasta_folha = MASCARAS_DIR / classe / "folha"
    pasta_sintoma = MASCARAS_DIR / classe / "sintomas"

    for pasta in [
        pasta_overlay,
        pasta_folha,
        pasta_sintoma
    ]:
        pasta.mkdir(
            parents=True,
            exist_ok=True
        )

    for caminho in arquivos:

        # Abrir imagem original
        img = np.array(
            Image.open(caminho).convert("RGB")
        )

        # Segmentar a folha
        mask_folha = criar_mascara_folha(img)

        # Identificar região sintomática
        mask_sintoma = gerar_mascara_sintoma(
            img,
            mask_folha,
            LIMIAR_FINAL
        )

        # Calcular áreas
        area_folha = int(
            np.count_nonzero(mask_folha)
        )

        area_afetada = int(
            np.count_nonzero(mask_sintoma)
        )

        if area_folha == 0:
            raise ValueError(
                f"Folha não identificada: {caminho.name}"
            )

        # Calcular severidade
        severidade = (
            area_afetada / area_folha
        ) * 100

        # Criar overlay
        overlay = criar_overlay(
            img,
            mask_sintoma
        )

        # Definir nomes dos arquivos
        nome = caminho.name

        saida_overlay = (
            pasta_overlay / f"{nome}_overlay.png"
        )

        saida_folha = (
            pasta_folha / f"{nome}_folha.png"
        )

        saida_sintoma = (
            pasta_sintoma / f"{nome}_sintomas.png"
        )

        # Salvar imagens processadas
        Image.fromarray(overlay).save(
            saida_overlay
        )

        Image.fromarray(mask_folha).save(
            saida_folha
        )

        Image.fromarray(mask_sintoma).save(
            saida_sintoma
        )

        # Verificação automática da máscara
        proporcao_folha = (
            area_folha / mask_folha.size
        )

        cantos = [
            mask_folha[0, 0],
            mask_folha[0, -1],
            mask_folha[-1, 0],
            mask_folha[-1, -1]
        ]

        avisos = []

        if proporcao_folha > 0.90:
            avisos.append(
                "Máscara ocupa mais de 90% da imagem"
            )

        if sum(v > 0 for v in cantos) >= 2:
            avisos.append(
                "Máscara alcança múltiplos cantos"
            )

        # Registrar os resultados
        resultados_severidade.append({

            "imagem": nome,

            "classe": classe,

            "area_folha_pixels": area_folha,

            "area_afetada_pixels": area_afetada,

            "severidade_percentual": severidade,

            "limiar": LIMIAR_FINAL,

            "caminho_original": str(caminho),

            "caminho_overlay": str(saida_overlay),

            "caminho_mascara_folha": str(saida_folha),

            "caminho_mascara_sintoma": str(saida_sintoma),

            "avisos": "; ".join(avisos)
        })

    print(
        f"{classe}: {len(arquivos)} imagens processadas."
    )

# Transformar resultados em tabela
df_severidade = pd.DataFrame(
    resultados_severidade
)

print("\nTotal processado:", len(df_severidade))

display(df_severidade.head())

# Resultados Estatísticos Severidade

In [ ]:
# Conferir quantidade de imagens
assert len(df_severidade) == 60

print("Imagens processadas por classe:")
print(df_severidade.groupby("classe").size())

# Resultados individuais
df_severidade.to_csv(
    SEVERIDADE_DIR / "resultados_severidade.csv",
    index=False,
    encoding="utf-8-sig"
)

df_severidade.to_excel(
    SEVERIDADE_DIR / "resultados_severidade.xlsx",
    index=False
)

# Cálculo estatísticas descritivas
estatisticas = (
    df_severidade
    .groupby("classe")["severidade_percentual"]
    .agg(
        quantidade="count",
        media="mean",
        mediana="median",
        minimo="min",
        maximo="max",
        desvio_padrao="std"
    )
    .round(2)
    .reset_index()
)

print("\nEstatísticas da severidade:")

display(estatisticas)

# Salvar estatísticas
estatisticas.to_csv(
    SEVERIDADE_DIR / "estatisticas_severidade.csv",
    index=False,
    encoding="utf-8-sig"
)

estatisticas.to_excel(
    SEVERIDADE_DIR / "estatisticas_severidade.xlsx",
    index=False
)

# Gráfico comparativo
dados_grafico = [
    df_severidade.loc[
        df_severidade["classe"] == classe,
        "severidade_percentual"
    ].dropna().values
    for classe in classes
]

plt.figure(figsize=(9, 6))

plt.boxplot(dados_grafico)

plt.xticks(
    range(1, len(classes) + 1),
    classes
)

plt.xlabel("Classe")
plt.ylabel("Severidade estimada (%)")

plt.title(
    "Distribuição da severidade por classe"
)

plt.ylim(bottom=0)

plt.tight_layout()

plt.savefig(
    FIGURAS_DIR / "boxplot_severidade.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nTabelas e gráfico salvos com sucesso!")

*Análise visual dos resultados de segmentação*

In [ ]:
fig, axes = plt.subplots(
    3,
    4,
    figsize=(14, 11)
)

for i, classe in enumerate(classes):

    linha = df_severidade[
        df_severidade["classe"] == classe
    ].iloc[0]

    original = Image.open(
        linha["caminho_original"]
    ).convert("RGB")

    folha = Image.open(
        linha["caminho_mascara_folha"]
    )

    sintomas = Image.open(
        linha["caminho_mascara_sintoma"]
    )

    overlay = Image.open(
        linha["caminho_overlay"]
    )

    imagens = [
        original,
        folha,
        sintomas,
        overlay
    ]

    titulos = [
        "Imagem original",
        "Máscara foliar",
        "Região sintomática",
        f"Severidade: {linha['severidade_percentual']:.2f}%"
    ]

    for j in range(4):

        if j in [1, 2]:

            axes[i, j].imshow(
                imagens[j],
                cmap="gray"
            )

        else:

            axes[i, j].imshow(imagens[j])

        axes[i, j].set_title(titulos[j])
        axes[i, j].axis("off")

    axes[i, 0].set_ylabel(
        classe,
        fontsize=12
    )

plt.tight_layout()

plt.savefig(
    FIGURAS_DIR / "etapas_processamento_severidade.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

*Conferência dos 60 overlays*

In [ ]:
for classe in classes:

    dados_classe = df_severidade[
        df_severidade["classe"] == classe
    ].reset_index(drop=True)

    fig, axes = plt.subplots(
        4,
        5,
        figsize=(15, 13)
    )

    for i, (_, linha) in enumerate(
        dados_classe.iterrows()
    ):

        overlay = Image.open(
            linha["caminho_overlay"]
        )

        ax = axes.ravel()[i]

        ax.imshow(overlay)

        ax.set_title(
            f"{linha['imagem']}\n"
            f"{linha['severidade_percentual']:.2f}%",
            fontsize=8
        )

        ax.axis("off")

    plt.suptitle(
        f"Resultados de segmentação - {classe}",
        fontsize=15
    )

    plt.tight_layout()

    plt.savefig(
        FIGURAS_DIR / f"conferencia_{classe}.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

# Imagens com alertas automáticos
revisar = df_severidade[
    df_severidade["avisos"] != ""
]

print(
    "Imagens com alertas de segmentação:",
    len(revisar)
)

display(
    revisar[
        [
            "imagem",
            "classe",
            "severidade_percentual",
            "avisos"
        ]
    ]
)

print("\nFiguras de conferência geradas!")